In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub
from langsmith import Client

C:\Users\teler\AppData\Local\Temp\ipykernel_10064\1813623027.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
# from langchain.agents import create_react_agent, AgentExecutor

from langchain.agents import create_agent

In [44]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY=os.getenv("GROQ_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")

In [45]:
search_tool = TavilySearchResults(max_results=3)

In [46]:
search_tool.invoke("What is the capital of France?")

[{'title': 'What is the capital city of France?',
  'url': 'https://www.facebook.com/groups/376100836095453/posts/2780652808973565',
  'content': "The capital of France is Paris, the country's largest city and main cultural and commercial centre. France, including its overseas territories,",
  'score': 0.9409691},
 {'title': 'What is the capital city of France and why is it famous?',
  'url': 'https://brainly.in/question/62118432',
  'content': "The capital city of France is Paris. It's famous for several reasons: Cultural heritage – Paris is home to iconic landmarks like the Eiffel",
  'score': 0.90791017},
 {'title': 'France | History, Maps, Flag, Population, Cities, Capital, & Facts',
  'url': 'https://www.britannica.com/place/France',
  'content': "Historically, the Francien dialect became the official language in 1539, eventually replacing Latin and other dialects. Although regional languages were once discouraged, they have been reintroduced in some schools because several have m

In [6]:
search_tool = TavilySearchResults(max_results=2)

In [7]:
search_tool.invoke("Give me the latest news about AI in 2026.")

[{'title': 'The Latest AI News and Breakthroughs That Matter Most | News',
  'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': "Date: July 2, 2026\n\nSummary: Anthropic is in early discussions with Samsung to develop a custom AI chip, joining competitors seeking to reduce reliance on Nvidia. The move follows OpenAI's announcement of its own custom inference processor built with Broadcom. Multiple AI companies now pursue specialized silicon to optimize performance and address compute constraints.\n\nSource:TechCrunch\n\n### (AI Update) New York Enacts First-of-Its-Kind Law Requiring AI-Generated Ad Disclosures\n\nDate: July 2, 2026 [...] Date: May 21, 2026\n\nSummary: SoundHound AI announced the acquisition of conversational AI platform LivePerson, combining its voice and audio recognition capabilities with LivePerson's digital messaging products. Management has set 2027 revenue targets of $350 million to $400 million and identified a $500 million cross-sel

In [47]:
llm=ChatGroq(model="openai/gpt-oss-120b")

In [48]:
response = llm.invoke("What is the capital of India?")

response

AIMessage(content='The capital of India is **New\u202fDelhi**.', additional_kwargs={'reasoning_content': 'The user asks a simple factual question: "What is the capital of India?" The answer: New Delhi. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 78, 'total_tokens': 124, 'completion_time': 0.097425126, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.003716715, 'prompt_tokens_details': None, 'queue_time': 0.396636215, 'total_time': 0.101141841}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15498-63f4-7d11-8043-1193e182c8ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 46, 'total_tokens': 124, 'output_token_details': {'reasoning': 26}})

In [49]:
client = Client()

prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [50]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [51]:
prompt.template

'Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'

In [52]:
tools = [search_tool]

In [ ]:
# Custome Tool

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [54]:
tools = [search_tool, get_weather_data]

In [55]:
## Creating the agent using the create_agent function

agent = create_agent(
    model=llm, 
    tools=tools,
    system_prompt=prompt.template
)

In [56]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the capital of India and then find its current weather"}
    ]
})

In [ ]:
print(result["messages"][-1].content)

Question: Find the capital of India and then find its current weather  
Thought: I have retrieved the current weather for New Delhi, which is the capital of India.  
Final Answer: The capital of India is **New Delhi**. Its current weather is a **sandstorm** with a temperature of **30 °C**, humidity of **21 %**.


Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: lsv2_********************************************55
Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.l

In [43]:
print(result["messages"])

[HumanMessage(content='What was the final score of the Super Bowl this year?', additional_kwargs={}, response_metadata={}, id='9816e29c-04f7-4ee3-b83f-7aa027b28c2b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What was the final score of the Super Bowl this year?" We need to find the final score of the Super Bowl this year. As of date 2026-10-19, "this year" likely refers to the most recent Super Bowl that occurred in 2026 (Super Bowl LVII was in 2023, LVIII in 2024, LIX in 2025, LX in 2026?). Actually, Super Bowl numbers: SB LV (2021), LVI (2022), LVII (2023), LVIII (2024), LIX (2025), LX (2026). The most recent Super Bowl as of early 2026 would be Super Bowl LX (2026) which likely took place in February 2026. Need the final score. Let\'s search.', 'tool_calls': [{'id': 'fc_6ab1e754-c68d-40c7-8145-a77542b66b9f', 'function': {'arguments': '{"query":"Super Bowl LX final score 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, respo